In [70]:
import torch
import math
import torch.nn as nn
from torch.nn import functional as F
from scipy.optimize import linear_sum_assignment
from collections import defaultdict

In [71]:
out_cls = torch.randn(4, 2, 25, 21)
out_bbox = torch.randn(4, 2, 25, 4)

In [72]:
print(out_cls.shape)
print(out_bbox.shape)

torch.Size([4, 2, 25, 21])
torch.Size([4, 2, 25, 4])


In [73]:
cls_idx_output = out_cls[0]
out_bbox_output = out_bbox[0]

In [74]:
print(cls_idx_output.shape)
print(out_bbox_output.shape)

torch.Size([2, 25, 21])
torch.Size([2, 25, 4])


In [75]:
class_prob = cls_idx_output.reshape(-1, 21).softmax(dim=-1)

In [76]:
for row in class_prob:
    print(row)

tensor([0.0171, 0.0186, 0.0172, 0.1204, 0.1268, 0.0161, 0.0039, 0.0096, 0.0062,
        0.0417, 0.0451, 0.0293, 0.0341, 0.0121, 0.0157, 0.1241, 0.2100, 0.0876,
        0.0101, 0.0464, 0.0079])
tensor([0.1155, 0.0107, 0.1183, 0.1321, 0.0161, 0.0075, 0.0728, 0.0337, 0.0528,
        0.0043, 0.0559, 0.0277, 0.0099, 0.0237, 0.0374, 0.0878, 0.0420, 0.0148,
        0.0891, 0.0200, 0.0278])
tensor([0.0093, 0.1096, 0.1127, 0.0255, 0.0482, 0.0344, 0.2013, 0.0426, 0.0149,
        0.0187, 0.0157, 0.0342, 0.0161, 0.0734, 0.0369, 0.0471, 0.0082, 0.0343,
        0.0489, 0.0531, 0.0149])
tensor([0.0319, 0.1039, 0.1180, 0.0331, 0.0213, 0.2863, 0.0403, 0.0240, 0.0144,
        0.0480, 0.0341, 0.0438, 0.0114, 0.0176, 0.0412, 0.0637, 0.0382, 0.0092,
        0.0022, 0.0081, 0.0092])
tensor([0.0500, 0.0011, 0.0257, 0.0053, 0.0034, 0.0091, 0.0693, 0.0687, 0.0269,
        0.0114, 0.0360, 0.2729, 0.0615, 0.0040, 0.0246, 0.0275, 0.1246, 0.0179,
        0.0223, 0.0071, 0.1308])
tensor([0.0425, 0.0339, 0.0839, 0.0

In [77]:
class_prob.shape

torch.Size([50, 21])

In [78]:
pred_boxes = out_bbox_output.reshape((-1, 4))

In [79]:
pred_boxes.shape

torch.Size([50, 4])

In [80]:
target_labels_1 = torch.arange(start=0, end=11, step=1)
target_labels_2 = torch.arange(start=0, end=5, step=1)
target_labels = torch.cat((target_labels_1, target_labels_2), dim=-1)

In [81]:
target_labels

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10,  0,  1,  2,  3,  4])

In [82]:
target_labels.shape

torch.Size([16])

In [83]:
cost_classification = -class_prob[:, target_labels]

In [85]:
cost_classification.shape

torch.Size([50, 16])

In [84]:
cost_classification

tensor([[-0.0171, -0.0186, -0.0172, -0.1204, -0.1268, -0.0161, -0.0039, -0.0096,
         -0.0062, -0.0417, -0.0451, -0.0171, -0.0186, -0.0172, -0.1204, -0.1268],
        [-0.1155, -0.0107, -0.1183, -0.1321, -0.0161, -0.0075, -0.0728, -0.0337,
         -0.0528, -0.0043, -0.0559, -0.1155, -0.0107, -0.1183, -0.1321, -0.0161],
        [-0.0093, -0.1096, -0.1127, -0.0255, -0.0482, -0.0344, -0.2013, -0.0426,
         -0.0149, -0.0187, -0.0157, -0.0093, -0.1096, -0.1127, -0.0255, -0.0482],
        [-0.0319, -0.1039, -0.1180, -0.0331, -0.0213, -0.2863, -0.0403, -0.0240,
         -0.0144, -0.0480, -0.0341, -0.0319, -0.1039, -0.1180, -0.0331, -0.0213],
        [-0.0500, -0.0011, -0.0257, -0.0053, -0.0034, -0.0091, -0.0693, -0.0687,
         -0.0269, -0.0114, -0.0360, -0.0500, -0.0011, -0.0257, -0.0053, -0.0034],
        [-0.0425, -0.0339, -0.0839, -0.0429, -0.0840, -0.0232, -0.0103, -0.0946,
         -0.0295, -0.0518, -0.0627, -0.0425, -0.0339, -0.0839, -0.0429, -0.0840],
        [-0.0248, -0.0